# CHAMMI-75, opened cold

This notebook opens the store using nothing but the store itself: no ingest code and no copy of the metadata CSV or the guidance zip. Everything below comes from the store's own metadata, its index and its pixels.

The repo is `earthmover-biology/chammi-75`. You need an Arraylake login with read access (`al auth login`) and the PNG codec (`pip install imagecodecs-zarr`). The store holds the studies of the CHAMMI-75 training set whose license is CC BY 4.0 or CC0 1.0. Its pixels are the dataset's own PNGs in `s3://chammi-data/`, read anonymously.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr
from arraylake import Client

repo = Client().get_repo("earthmover-biology/chammi-75")
root = zarr.open_group(repo.readonly_session("main").store, mode="r")

print(root.attrs["title"])
print(root.attrs["citation"])
print(root.attrs["contents"])

## Which studies, under which license

The root lists every study of the release with its license, and the studies the store leaves out with the reason. Each study's group states its own license again.

In [ ]:
licenses = root.attrs["licenses"]
print("source:", licenses["source"])
print(pd.Series(licenses["included"]).value_counts().to_string())
print("excluded:", pd.Series(licenses["excluded"]).value_counts().to_string(), sep="\n")

## Finding images

`index/` has one row per image: its study, its stack and its row in the stack, and every annotation the source states. It is small enough to load whole.

In [ ]:
index = pd.DataFrame({name: root[f"index/{name}"][:] for name in root["index"].array_keys()})
print(f"{len(index):,} images in {index['stack'].nunique():,} stacks")
index.groupby(["study", "license"]).size().rename("images").reset_index().head(10)

## One image, all of its channels

A stack holds the images of one study that share a size and a set of channels. Each channel slot is an OME-Zarr image, `<stack>/ch<slot>`, with axes `(image, y, x)`. Row `r` of every channel image is the same image, so reading one image means reading row `r` of each slot listed in `index/channels`. Each chunk is one PNG.

In [ ]:
pick = index[(index["n_channels"] > 1) & index["has_positions"]].iloc[0]
slots = pick["channels"].split("-")
planes = [root[f"{pick['stack']}/ch{slot}/0"][pick["row"]] for slot in slots]
types = pick["channel_types"].split(",")
print(pick[["image_id", "stack", "row", "width", "height", "channels", "channel_types"]].to_string())

fig, axes = plt.subplots(1, len(planes), figsize=(4 * len(planes), 4))
for ax, plane, slot, kind in zip(np.atleast_1d(axes), planes, slots, types):
    ax.imshow(plane, cmap="gray")
    ax.set_title(f"ch{slot}: {kind}")
    ax.axis("off")

## Its cells

`cells/` sits on the same image axis as `index/`: index row `i`'s cells are rows `image_start[i]` to `image_start[i + 1]` of `cells/position`. Where the source file's first entry was the background, it is kept apart in `cells/background`.

In [ ]:
i = int(pick.name)
start, end = root["cells/image_start"][i : i + 2]
xy = root["cells/position"][start:end]
print(f"{len(xy)} cells, segmentation {pick['segmentation']}, background entry moved: {pick['background_dropped']}")

plt.figure(figsize=(6, 6))
plt.imshow(planes[0], cmap="gray")
plt.scatter(xy[:, 0], xy[:, 1], s=12, c="tab:orange")
plt.axis("off");